# AIOps: Log Anomaly Detection (Isolation Forests)

In [ ]:
# Day 44 — AIOps: Log Anomaly Detection
#
# The Scenario:
# Your microservices generate 1,000,000 lines of logs per day. 
# Normally, the API responds in 50ms, the DB takes 10ms, and 99% of requests are HTTP 200.
# 
# Suddenly, a weird deployment happens. No errors are thrown, but the API now takes 200ms
# and DB takes 40ms. A human won't notice this looking at a wall of text until users complain.
#
# How do we detect "weirdness" automatically?
# We use an Unsupervised Machine Learning algorithm called the Isolation Forest.
#
# Today we'll cover:
#   1. What is an Isolation Forest?
#   2. Feature Engineering for Logs
#   3. Training the Model to find Anomalies
#
# Install: pip install scikit-learn pandas numpy

### 1. Generating Mock Log Data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
import matplotlib.pyplot as plt

# Let's generate 1000 normal log events
np.random.seed(42)
normal_api_latency = np.random.normal(50, 10, 1000) # mean 50ms, std 10ms
normal_db_latency = np.random.normal(15, 5, 1000)   # mean 15ms, std 5ms

# Let's generate 50 ANOMALOUS log events (the weird deployment)
anomaly_api_latency = np.random.normal(180, 20, 50) # High latency
anomaly_db_latency = np.random.normal(60, 15, 50)   # High latency

# Combine them into a single dataframe
api_latency = np.concatenate([normal_api_latency, anomaly_api_latency])
db_latency = np.concatenate([normal_db_latency, anomaly_db_latency])

logs_df = pd.DataFrame({
    'api_latency_ms': api_latency,
    'db_latency_ms': db_latency
})

# Shuffle the data so anomalies are mixed in
logs_df = logs_df.sample(frac=1).reset_index(drop=True)

print("Sample of our logs:")
print(logs_df.head())

### 2. The Isolation Forest

In [ ]:
# Isolation Forest is perfect for AIOps.
# It works by randomly "cutting" the data.
# Normal data points are clustered together, so it takes MANY cuts to isolate them.
# Anomalies are far away, so it takes very FEW cuts to isolate them.
# If a log line is easily isolated, the model flags it as an anomaly!

# 1. Initialize the model
# contamination=0.05 means we guess roughly 5% of our logs are anomalous
iso_forest = IsolationForest(contamination=0.05, random_state=42)

# 2. Fit and Predict
# In unsupervised learning, we don't have 'y' (labels). We just pass the features 'X'!
logs_df['anomaly_score'] = iso_forest.fit_predict(logs_df[['api_latency_ms', 'db_latency_ms']])

# The model returns 1 for NORMAL, and -1 for ANOMALY
logs_df['is_anomaly'] = logs_df['anomaly_score'] == -1

print(f"Total logs: {len(logs_df)}")
print(f"Detected anomalies: {logs_df['is_anomaly'].sum()}")

# Let's look at a detected anomaly
print("\nSample Anomaly:")
print(logs_df[logs_df['is_anomaly']].head(1))

### 3. Visualizing the Anomalies

In [ ]:
plt.figure(figsize=(10, 6))

# Plot normal logs in blue
normal = logs_df[~logs_df['is_anomaly']]
plt.scatter(normal['api_latency_ms'], normal['db_latency_ms'], c='blue', label='Normal', alpha=0.5)

# Plot anomalous logs in red
anomalies = logs_df[logs_df['is_anomaly']]
plt.scatter(anomalies['api_latency_ms'], anomalies['db_latency_ms'], c='red', label='Anomaly', marker='x')

plt.title("AIOps: Isolation Forest Log Anomaly Detection")
plt.xlabel("API Latency (ms)")
plt.ylabel("DB Latency (ms)")
plt.legend()
plt.grid(True)

plt.savefig('log_anomalies.png')
print("Plot saved to log_anomalies.png. Open it to see how well it separated the weird logs!")

### Detecting Anomalies in Real Datasets (`datasets/`)

In [ ]:
# Load the real 720-hour telemetry (contains 8 injected CPU+latency anomalies!)
import pandas as pd
from sklearn.ensemble import IsolationForest

df = pd.read_csv("datasets/server_metrics_timeseries.csv")
iso = IsolationForest(contamination=0.015, random_state=42)
df["anomaly"] = iso.fit_predict(df[["cpu_pct", "latency_ms"]])
anomalies = df[df["anomaly"] == -1]
print(f"Detected {len(anomalies)} anomalous hours:")
print(anomalies[["timestamp", "cpu_pct", "latency_ms"]].head(10))

In [ ]:
# Practice Questions:
#
# 1. Why do we use an Unsupervised model (Isolation Forest) for logs instead of 
#    a Supervised model (like Random Forest)?
# 2. If you set `contamination=0.20`, what happens to the number of detected anomalies?

### DevOps / AI-MLOps Practice

In [ ]:
# Exercise 1: The Alerting Pipeline
# Write a python script that reads a live JSON log file. For every new line, it extracts
# the latencies and passes it to the `iso_forest.predict()` function.
# If it returns -1, it sends a Slack alert saying "Anomalous log detected!"

# Why this matters for me as a DevOps/MLOps engineer:
# 1. Alert Fatigue: Traditional DevOps relies on hardcoded thresholds (e.g., ALERT if API > 100ms).
#    This causes "Alert Fatigue" because APIs naturally slow down under heavy legitimate load.
# 2. Machine Learning thresholds dynamically adjust to what is "normal" for your system's current state.
# 3. Tools like Datadog Watchdog use Isolation Forests under the hood. Now you know how to build it yourself!